# Week 7: Query Intent Classification

This notebook reviews the manually labeled query set and the held-out performance of the language-based intent classifier.


In [1]:
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.real_estate_nlp.query_intent_classifier import QueryIntentClassifier
from src.real_estate_nlp.query_parser import QueryParser

labels_path = Path("../data/processed/query_intent_labels.json")
dev_results_path = Path("../data/processed/query_intent_dev_results.json")
results_path = Path("../data/processed/query_intent_eval_results.json")
model_dir = Path("../data/models/query_intent")

---

## 1. Artifacts Loading

We first load the labeled query set.


In [2]:
items = json.loads(labels_path.read_text())["items"]
dev_results = json.loads(dev_results_path.read_text())
results = json.loads(results_path.read_text())
queries = pd.DataFrame(items)
queries.shape


(504, 6)

The label set is balanced by design. The development split was used for model refinement. The test split was used for evaluation.


In [3]:
label_summary = (
    queries.groupby(["split", "label"], sort=False)
    .size()
    .unstack(fill_value=0)
)
label_summary["total"] = label_summary.sum(axis=1)
label_summary


label,browsing,researching,high_intent_inquiry,total
split,,,,
train,120,120,120,360
dev,24,24,24,72
test,24,24,24,72


---

## 2. Held-Out Evaluation

The tables below report the final test result.


In [5]:
accuracy = pd.DataFrame(
    {
        "metric": ["Development accuracy", "Final held-out accuracy"],
        "value": [dev_results["accuracy"], results["accuracy"]],
    }
)
display(accuracy.round(3))

per_label = pd.DataFrame(results["per_label"]).T
per_label[["precision", "recall", "f1", "support"]].round(3)


,metric,value
0,Development accuracy,0.986
1,Final held-out accuracy,0.958


,precision,recall,f1,support
browsing,1.000,0.875,0.933,24.0
researching,1.000,1.000,1.000,24.0
high_intent_inquiry,0.889,1.000,0.941,24.0


---

## 3. Confidence and Errors

Confidence is the largest calibrated class probability.


In [7]:
confidence = pd.DataFrame(results["confidence"]["bins"])
confidence["accuracy"] = confidence["accuracy"].round(3)
confidence


,range,count,accuracy
0,0.00-0.60,8,0.625
1,0.60-0.75,12,1.000
2,0.75-0.90,25,1.000
3,0.90-1.00,27,1.000


In [8]:
errors = pd.DataFrame(results["errors"])
errors[["query", "expected", "predicted", "confidence"]].round({"confidence": 3})


,query,expected,predicted,confidence
0,I am just looking at Long Beach listings,browsing,high_intent_inquiry,0.557
1,could you show Pasadena condos under 800k,browsing,high_intent_inquiry,0.535
2,I am browsing San Diego townhomes,browsing,high_intent_inquiry,0.380


---

## 4. Parser Integration

The two intent fields remain independent. `intent` describes the parser task, while `language_intent` reflects the wording of the search query.


In [9]:
classifier = QueryIntentClassifier.load(model_dir)
parser = QueryParser(intent_classifier=classifier)

examples = [
    "homes with pools in Irvine",
    "which area has lower HOA fees near Irvine",
    "schedule a tour of move-in ready homes in Irvine this weekend",
]

integration = []
for query in examples:
    parsed = parser.parse(query)
    integration.append(
        {
            "query": query,
            "parser_intent": parsed["intent"],
            "language_intent": parsed["language_intent"]["label"],
            "confidence": parsed["language_intent"]["confidence"],
            "hard_filters": parsed["hard_filters"],
            "soft_signals": parsed["soft_signals"],
        }
    )

display(pd.DataFrame(integration).round({"confidence": 3}))


,query,parser_intent,language_intent,confidence,hard_filters,soft_signals
0,homes with pools in Irvine,location_search,browsing,0.943,{'city': 'Irvine'},{}
1,which area has lower HOA fees near Irvine,location_search,researching,0.835,{'city': 'Irvine'},{}
2,schedule a tour of move-in ready homes in Irvi...,condition_search,high_intent_inquiry,0.755,{'city': 'Irvine'},{'condition': ['move-in ready']}
